In [2]:
# Verwendete Bibliotheken
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
import time
import difflib
import statsmodels.api as sm
import statsmodels.formula.api as smf
import random
import numpy as np
from itertools import combinations
import math
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns


In [1]:
model_df = pd.read_excel("model_df.xlsx")

In [3]:
# Poisson-Regressionsmodell mit log-Link-Funktion zur Vorhersage der Tore
poisson_model = smf.glm(
    formula="Goals ~ Elo_Diff + Home",  # Modellformel: Zielvariable ist 'Goals'
    data=model_df,                      # Trainingsdaten
    family=sm.families.Poisson()        # Poisson-Verteilung (geeignet für Zählvariablen)
).fit()

# Modellzusammenfassung anzeigen (Koeffizienten, Standardfehler, Signifikanz usw.)
print(poisson_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  Goals   No. Observations:                 1500
Model:                            GLM   Df Residuals:                     1497
Model Family:                 Poisson   Df Model:                            2
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -2249.0
Date:                Tue, 16 Sep 2025   Deviance:                       1736.5
Time:                        09:55:44   Pearson chi2:                 1.52e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.2146
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.2289      0.032      7.090      0.0

In [4]:
# Teams und ihre Topfzuordnung gemäß UEFA-Auslosung 2023/24
teams_with_pots = [
    # Pot 1
    ("Man City", 1), ("Sevilla", 1), ("Barcelona", 1), ("Napoli", 1),
    ("Bayern", 1), ("Paris SG", 1), ("Benfica", 1), ("Feyenoord", 1),
    # Pot 2
    ("Real Madrid", 2), ("Man United", 2), ("Inter", 2), ("Dortmund", 2),
    ("Atlético", 2), ("RB Leipzig", 2), ("Porto", 2), ("Arsenal", 2),
    # Pot 3
    ("Shakhtar", 3), ("Salzburg", 3), ("Milan", 3), ("Braga", 3),
    ("PSV", 3), ("Lazio", 3), ("Crvena Zvezda", 3), ("Copenhagen", 3),
    # Pot 4
    ("Young Boys", 4), ("Real Sociedad", 4), ("Galatasaray", 4), ("Celtic", 4),
    ("Newcastle", 4), ("Union Berlin", 4), ("Antwerp", 4), ("Lens", 4),
]

# Elo-Ratings der Teams 
elo_dict = {
    "Man City": 2091, "Sevilla": 1733, "Barcelona": 1878, "Napoli": 1892,
    "Bayern": 1940, "Paris SG": 1801, "Benfica": 1806, "Feyenoord": 1753,
    "Real Madrid": 1922, "Man United": 1847, "Inter": 1909, "Dortmund": 1841,
    "Atlético": 1838, "RB Leipzig": 1841, "Porto": 1825, "Arsenal": 1932,
    "Shakhtar": 1591, "Salzburg": 1730, "Milan": 1841, "Braga": 1662,
    "PSV": 1778, "Lazio": 1760, "Crvena Zvezda": 1573, "Copenhagen": 1632,
    "Young Boys": 1630, "Real Sociedad": 1772, "Galatasaray": 1696,
    "Celtic": 1634, "Newcastle": 1872, "Union Berlin": 1735,
    "Antwerp": 1656, "Lens": 1709
}

# DataFrame mit Clubs, Töpfen und zugeordneten Elo-Werten
df_pots = pd.DataFrame(teams_with_pots, columns=["Club", "Pot"])
df_pots["Elo"] = df_pots["Club"].map(elo_dict)


In [5]:
df_pots

,Club,Pot,Elo
0,Man City,1,2091
1,Sevilla,1,1733
2,Barcelona,1,1878
3,Napoli,1,1892
4,Bayern,1,1940
5,Paris SG,1,1801
6,Benfica,1,1806
7,Feyenoord,1,1753
8,Real Madrid,2,1922
9,Man United,2,1847


In [94]:
# Zuweisung der Clubs zu ihren Ländern gemäß UEFA-Saison 2023/24
club_to_country = {
    "Man City": "ENG", "Sevilla": "ESP", "Barcelona": "ESP", "Napoli": "ITA",
    "Bayern": "GER", "Paris SG": "FRA", "Benfica": "POR", "Feyenoord": "NED",
    "Real Madrid": "ESP", "Man United": "ENG", "Inter": "ITA", "Dortmund": "GER",
    "Atlético": "ESP", "RB Leipzig": "GER", "Porto": "POR", "Arsenal": "ENG",
    "Shakhtar": "UKR", "Salzburg": "AUT", "Milan": "ITA", "Braga": "POR",
    "PSV": "NED", "Lazio": "ITA", "Crvena Zvezda": "SRB", "Copenhagen": "DEN",
    "Young Boys": "SUI", "Real Sociedad": "ESP", "Galatasaray": "TUR", "Celtic": "SCO",
    "Newcastle": "ENG", "Union Berlin": "GER", "Antwerp": "BEL", "Lens": "FRA"
}

# Ergänze die Herkunftsländer der Teams im DataFrame (für spätere UEFA-Regelprüfung bei Auslosung)
df_pots["Country"] = df_pots["Club"].map(club_to_country)

In [95]:
# Versuche bis zu max_attempts-mal eine gültige Gruppenauslosung zu erstellen
max_attempts = 1000  # Sicherheitsschleife, falls keine gültige Kombination gefunden wird

for attempt in range(max_attempts):
    try:
        # Leere Gruppen initialisieren (A–H)
        groups = {f"Group {chr(65+i)}": [] for i in range(8)}

        # Schleife über die vier Töpfe
        for pot_num in range(1, 5):
            pot_teams = df_pots[df_pots["Pot"] == pot_num].copy()
            pot_teams = pot_teams.sample(frac=1).to_dict("records")  # Teams im Topf zufällig mischen

            for team in pot_teams:
                assigned = False
                group_names = list(groups.keys())
                random.shuffle(group_names)  # Gruppen zufällig durchgehen

                for group_name in group_names:
                    existing_countries = [t["Country"] for t in groups[group_name]]
                    
                    # Bedingung: Max. 1 Team pro Land und Topfgröße nicht überschreiten
                    if len(groups[group_name]) < pot_num and team["Country"] not in existing_countries:
                        groups[group_name].append(team)
                        assigned = True
                        break  # Team erfolgreich zugewiesen

                if not assigned:
                    # Kein passender Platz → Auslosung ungültig → Neustart
                    raise ValueError(f"Kein gültiger Platz für {team['Club']}")

        # Wenn alle Zuweisungen gültig waren → Abbruch der Schleife
        break

    except ValueError:
        continue  # Wiederhole mit neuer zufälliger Reihenfolge

else:
    # Nach max_attempts keine gültige Auslosung → Fehler
    raise RuntimeError("Es konnte keine gültige Gruppenauslosung erstellt werden.")

# Umwandlung in DataFrame zur Weiterverarbeitung
group_assignments = []
for group, teams in groups.items():
    for team in teams:
        group_assignments.append({
            "Group": group,
            "Club": team["Club"],
            "Country": team["Country"],
            "Pot": team["Pot"],
            "Elo": team["Elo"]
        })

df_group_draw = pd.DataFrame(group_assignments).sort_values(by=["Group", "Pot"]).reset_index(drop=True)

# Ausgabe zur Kontrolle
print("UEFA-konforme Gruppenauslosung erfolgreich abgeschlossen:")
display(df_group_draw)

UEFA-konforme Gruppenauslosung erfolgreich abgeschlossen:


,Group,Club,Country,Pot,Elo
0,Group A,Sevilla,ESP,1,1733
1,Group A,Man United,ENG,2,1847
2,Group A,Shakhtar,UKR,3,1591
3,Group A,Young Boys,SUI,4,1630
4,Group B,Barcelona,ESP,1,1878
5,Group B,Porto,POR,2,1825
6,Group B,Milan,ITA,3,1841
7,Group B,Union Berlin,GER,4,1735
8,Group C,Bayern,GER,1,1940
9,Group C,Inter,ITA,2,1909


In [101]:
random.seed(42)
np.random.seed(42)
# Geschätzte Koeffizienten aus Poisson-Modell (basierend auf vorherigem Training)
BETA_0 = 0.2289
BETA_ELO = 0.0020
BETA_HOME = 0.2019

# Erwartete Tore auf Basis von Elo-Differenz und Heimvorteil
def expected_goals(elo_diff, home):
    return math.exp(BETA_0 + BETA_ELO * elo_diff + BETA_HOME * home)

# Simuliere Gruppenphase (Hin- und Rückspiel zwischen allen Teams)
def simulate_group_stage(df_group_draw):
    results = []
    for group in df_group_draw["Group"].unique():
        teams = df_group_draw[df_group_draw["Group"] == group].reset_index(drop=True)
        for i in range(4):
            for j in range(i+1, 4):
                team1, team2 = teams.iloc[i], teams.iloc[j]
                
                # Hinspiel: team1 ist Heimteam
                lam1 = expected_goals(team1["Elo"] - team2["Elo"], 1)
                lam2 = expected_goals(team2["Elo"] - team1["Elo"], 0)
                g1 = np.random.poisson(lam1)
                g2 = np.random.poisson(lam2)
                results.append((group, team1["Club"], team2["Club"], g1, g2))
                
                # Rückspiel: team2 ist Heimteam
                lam1 = expected_goals(team2["Elo"] - team1["Elo"], 1)
                lam2 = expected_goals(team1["Elo"] - team2["Elo"], 0)
                g1 = np.random.poisson(lam1)
                g2 = np.random.poisson(lam2)
                results.append((group, team2["Club"], team1["Club"], g1, g2))
    
    return pd.DataFrame(results, columns=["Group", "Home", "Away", "Goals Home", "Goals Away"])

# Punkte & Tordifferenz in der Gruppenphase berechnen
def compute_group_points(matches):
    points = {}
    goal_diff = {}
    for _, row in matches.iterrows():
        h, a = row["Home"], row["Away"]
        gh, ga = row["Goals Home"], row["Goals Away"]

        # Initialisiere falls Team noch nicht in Dict ist
        points[h] = points.get(h, 0)
        points[a] = points.get(a, 0)
        goal_diff[h] = goal_diff.get(h, 0)
        goal_diff[a] = goal_diff.get(a, 0)

        # Punktevergabe
        if gh > ga:
            points[h] += 3
        elif gh < ga:
            points[a] += 3
        else:
            points[h] += 1
            points[a] += 1

        # Tordifferenz aktualisieren
        goal_diff[h] += gh - ga
        goal_diff[a] += ga - gh

    return points, goal_diff

# K.O.-Runde (z. B. Achtel-, Viertel-, Halbfinale): Heim- & Rückspiel
def simulate_knockout_round(teams, df_group_draw):
    random.shuffle(teams)
    next_round = []
    for i in range(0, len(teams), 2):
        t1, t2 = teams[i], teams[i+1]
        elo1 = df_group_draw[df_group_draw["Club"] == t1]["Elo"].values[0]
        elo2 = df_group_draw[df_group_draw["Club"] == t2]["Elo"].values[0]

        # Hinspiel: t1 Heim
        g1 = np.random.poisson(expected_goals(elo1 - elo2, 1))
        g2 = np.random.poisson(expected_goals(elo2 - elo1, 0))

        # Rückspiel: t2 Heim
        g3 = np.random.poisson(expected_goals(elo2 - elo1, 1))
        g4 = np.random.poisson(expected_goals(elo1 - elo2, 0))

        total1 = g1 + g4
        total2 = g2 + g3

        # Sieger basierend auf Gesamttoren
        if total1 > total2:
            next_round.append(t1)
        elif total2 > total1:
            next_round.append(t2)
        else:
            next_round.append(random.choice([t1, t2]))  # Zufall bei Gleichstand

    return next_round

# Finale simulieren – 1 Spiel auf neutralem Boden (kein Heimvorteil)
def simulate_final(team1, team2, df_group_draw):
    elo1 = df_group_draw[df_group_draw["Club"] == team1]["Elo"].values[0]
    elo2 = df_group_draw[df_group_draw["Club"] == team2]["Elo"].values[0]
    g1 = np.random.poisson(expected_goals(elo1 - elo2, 0))
    g2 = np.random.poisson(expected_goals(elo2 - elo1, 0))

    if g1 > g2:
        return team1
    elif g2 > g1:
        return team2
    return random.choice([team1, team2])  # Zufall bei Gleichstand

# ================================ Haupt-Simulation ================================

winner_counts = Counter()

# Simulation des gesamten Turniers 10.000 Mal
for sim in range(10000):
    # Gruppenphase simulieren
    group_matches = simulate_group_stage(df_group_draw)
    pts, gd = compute_group_points(group_matches)

    # Punktestand und Tordifferenz in Tabelle übernehmen
    teams_in_groups = df_group_draw.copy()
    teams_in_groups["Points"] = teams_in_groups["Club"].map(pts).fillna(0)
    teams_in_groups["GD"] = teams_in_groups["Club"].map(gd).fillna(0)

    # Top 2 jeder Gruppe kommen weiter
    qualified = []
    for group in teams_in_groups["Group"].unique():
        top2 = (
            teams_in_groups[teams_in_groups["Group"] == group]
            .sort_values(by=["Points", "GD"], ascending=False)
            .head(2)["Club"]
            .tolist()
        )
        qualified.extend(top2)

    # K.O.-Phase simulieren
    qf = simulate_knockout_round(qualified, df_group_draw)
    sf = simulate_knockout_round(qf, df_group_draw)
    finalists = simulate_knockout_round(sf, df_group_draw)
    winner = simulate_final(finalists[0], finalists[1], df_group_draw)

    # Gewinner speichern
    winner_counts[winner] += 1

# Turniergewinner zusammenfassen & anzeigen
results_df = pd.DataFrame.from_dict(winner_counts, orient="index", columns=["Wins"]).sort_values(by="Wins", ascending=False)
print("🏆 Turniersieger nach 10.000 Simulationen:")
display(results_df)

🏆 Turniersieger nach 10.000 Simulationen:


,Wins
Man City,4566
Bayern,874
Arsenal,824
Real Madrid,742
Inter,617
Napoli,421
Barcelona,324
Man United,270
Newcastle,247
RB Leipzig,165


## visualisierungen

In [105]:
# Gesamtanzahl aller Simulationen berechnen
total_simulations = results_df["Wins"].sum()

# Gewinnwahrscheinlichkeit (in Prozent) für jedes Team berechnen
results_df["Win Probability (%)"] = (results_df["Wins"] / total_simulations) * 100

# Index zurücksetzen und Spalte korrekt benennen
results_df = results_df.reset_index().rename(columns={"index": "Team"})

# DataFrame nach Gewinnwahrscheinlichkeit sortieren
results_df = results_df.sort_values("Win Probability (%)", ascending=False)

# Visualisierung: Balkendiagramm erstellen
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df, x="Team", y="Win Probability (%)", palette="Blues_d")

# Diagrammbeschriftung und Formatierung
plt.title("Gewinnwahrscheinlichkeit pro Team im alten CL-Format", fontsize=14, fontweight='bold')
plt.ylabel("Wahrscheinlichkeit (%)", fontweight='bold')
plt.xlabel("Team", fontweight='bold')
plt.xticks(rotation=45, ha='right')  # Teamnamen schräg darstellen
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)  # Horizontale Hilfslinien

plt.savefig("gewinnwahrscheinlichkeit_altes_format.pdf", format="pdf")

plt.show()


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

<Figure size 1200x600 with 0 Axes>

In [46]:
random.seed(42)
np.random.seed(42)
# Fortschrittszähler initialisieren
progress = {
    "Round of 16": Counter(),
    "Quarter-finals": Counter(),
    "Semi-finals": Counter(),
    "Final": Counter(),
    "Champion": Counter()
}

n_simulations = 100  # oder z. B. 10000

for sim in range(n_simulations):
    group_matches = simulate_group_stage(df_group_draw)
    pts, gd = compute_group_points(group_matches)

    teams_in_groups = df_group_draw.copy()
    teams_in_groups["Points"] = teams_in_groups["Club"].map(pts).fillna(0)
    teams_in_groups["GD"] = teams_in_groups["Club"].map(gd).fillna(0)

    qualified = []
    for group in teams_in_groups["Group"].unique():
        top2 = (
            teams_in_groups[teams_in_groups["Group"] == group]
            .sort_values(by=["Points", "GD"], ascending=False)
            .head(2)["Club"]
            .tolist()
        )
        qualified.extend(top2)

    for team in qualified:
        progress["Round of 16"][team] += 1

    qf = simulate_knockout_round(qualified, df_group_draw)
    for team in qf:
        progress["Quarter-finals"][team] += 1

    sf = simulate_knockout_round(qf, df_group_draw)
    for team in sf:
        progress["Semi-finals"][team] += 1

    finalists = simulate_knockout_round(sf, df_group_draw)
    for team in finalists:
        progress["Final"][team] += 1

    winner = simulate_final(finalists[0], finalists[1], df_group_draw)
    progress["Champion"][winner] += 1

# === Wahrscheinlichkeiten berechnen ===
teams = set()
for stage in progress:
    teams.update(progress[stage].keys())

elo_map = df_group_draw.set_index("Club")["Elo"].to_dict()

table = []
for team in sorted(teams):
    row = {
        "Team": team,
        "Elo": elo_map.get(team, None)
    }
    for stage in ["Round of 16", "Quarter-finals", "Semi-finals", "Final", "Champion"]:
        count = progress[stage].get(team, 0)
        percentage = (count / n_simulations) * 100
        row[stage] = "<1%" if percentage < 0.5 else f"{percentage:.0f}%"
    table.append(row)

df_probabilities = pd.DataFrame(table)

# === Sortieren nach Elo (stärkstes Team oben) ===
df_probabilities = (
    df_probabilities
    .sort_values(by="Elo", ascending=False)
    .reset_index(drop=True)
)[["Team", "Elo", "Round of 16", "Quarter-finals", "Semi-finals", "Final", "Champion"]]

# === MultiIndex für Gruppierung der Spaltenüberschriften ===
new_columns = pd.MultiIndex.from_tuples([
    ("", "Team"),
    ("Teamstärke", "Elo"),
    ("Wahrscheinlichkeit für Erreichen der K.-o.-Phase", "Achtelfinale"),
    ("Wahrscheinlichkeit für Erreichen der K.-o.-Phase", "Viertelfinale"),
    ("Wahrscheinlichkeit für Erreichen der K.-o.-Phase", "Halbfinale"),
    ("Wahrscheinlichkeit für Erreichen der K.-o.-Phase", "Finale"),
    ("Wahrscheinlichkeit für Erreichen der K.-o.-Phase", "Turniersieg")
])
df_probabilities.columns = new_columns

# === Styling-Funktion für grüne Zellen ===
def style_probabilities_multiindex(df):
    def color_map(val):
        if val == "<1%":
            val_num = 0.4
        else:
            val_num = float(val.strip('%'))
        green = int(255 - (val_num / 100) * 150)
        return f"background-color: rgb({green}, 255, {green});"

    probability_cols = df.columns[2:]
    
    return (
        df.style.format({col: lambda x: x for col in probability_cols})
          .applymap(color_map, subset=probability_cols)
          .set_properties(**{'text-align': 'center', 'font-weight': 'bold'})
          .set_table_styles([
              {"selector": "th", "props": [
                  ("background-color", "#f2f2f2"),
                  ("font-weight", "bold"),
                  ("text-align", "center")
              ]}
          ])
    )


# === Darstellung anzeigen ===
styled_df = style_probabilities_multiindex(df_probabilities)
display(styled_df)


In [47]:
html_str = styled_df.to_html()

with open("knockout_table.html", "w", encoding="utf-8") as f:
    f.write(html_str)

print("✅ Datei 'knockout_table.html' gespeichert – einfach öffnen & als PDF drucken.")


✅ Datei 'knockout_table.html' gespeichert – einfach öffnen & als PDF drucken.


### Old Format for Liverpool - for journalistic report

In [54]:
# Teams und ihre Topfzuordnung gemäß UEFA-Auslosung 2023/24
teams_with_pots = [
    # Pot 1
    ("Man City", 1), ("Sevilla", 1), ("Barcelona", 1), ("Napoli", 1),
    ("Bayern", 1), ("Paris SG", 1), ("Benfica", 1), ("Feyenoord", 1),
    # Pot 2
    ("Real Madrid", 2), ("Man United", 2), ("Inter", 2), ("Dortmund", 2),
    ("Atlético", 2), ("RB Leipzig", 2), ("Porto", 2), ("Arsenal", 2),
    # Pot 3
    ("Shakhtar", 3), ("Salzburg", 3), ("Milan", 3), ("Braga", 3),
    ("PSV", 3), ("Lazio", 3), ("Crvena Zvezda", 3), ("Copenhagen", 3),
    # Pot 4
    ("Young Boys", 4), ("Real Sociedad", 4), ("Galatasaray", 4), ("Celtic", 4),
    ("Liverpool", 4), ("Union Berlin", 4), ("Antwerp", 4), ("Lens", 4),
]

# Elo-Ratings der Teams 
# Liverpools elo rating on 13-08-2023
elo_dict = {
    "Man City": 2091, "Sevilla": 1733, "Barcelona": 1878, "Napoli": 1892,
    "Bayern": 1940, "Paris SG": 1801, "Benfica": 1806, "Feyenoord": 1753,
    "Real Madrid": 1922, "Man United": 1847, "Inter": 1909, "Dortmund": 1841,
    "Atlético": 1838, "RB Leipzig": 1841, "Porto": 1825, "Arsenal": 1932,
    "Shakhtar": 1591, "Salzburg": 1730, "Milan": 1841, "Braga": 1662,
    "PSV": 1778, "Lazio": 1760, "Crvena Zvezda": 1573, "Copenhagen": 1632,
    "Young Boys": 1630, "Real Sociedad": 1772, "Galatasaray": 1696,
    "Celtic": 1634, "Liverpool": 1946, "Union Berlin": 1735,
    "Antwerp": 1656, "Lens": 1709
}

# DataFrame mit Clubs, Töpfen und zugeordneten Elo-Werten
df_pots = pd.DataFrame(teams_with_pots, columns=["Club", "Pot"])
df_pots["Elo"] = df_pots["Club"].map(elo_dict)


In [55]:
df_pots

,Club,Pot,Elo
0,Man City,1,2091
1,Sevilla,1,1733
2,Barcelona,1,1878
3,Napoli,1,1892
4,Bayern,1,1940
5,Paris SG,1,1801
6,Benfica,1,1806
7,Feyenoord,1,1753
8,Real Madrid,2,1922
9,Man United,2,1847


In [56]:
# Zuweisung der Clubs zu ihren Ländern gemäß UEFA-Saison 2023/24
club_to_country = {
    "Man City": "ENG", "Sevilla": "ESP", "Barcelona": "ESP", "Napoli": "ITA",
    "Bayern": "GER", "Paris SG": "FRA", "Benfica": "POR", "Feyenoord": "NED",
    "Real Madrid": "ESP", "Man United": "ENG", "Inter": "ITA", "Dortmund": "GER",
    "Atlético": "ESP", "RB Leipzig": "GER", "Porto": "POR", "Arsenal": "ENG",
    "Shakhtar": "UKR", "Salzburg": "AUT", "Milan": "ITA", "Braga": "POR",
    "PSV": "NED", "Lazio": "ITA", "Crvena Zvezda": "SRB", "Copenhagen": "DEN",
    "Young Boys": "SUI", "Real Sociedad": "ESP", "Galatasaray": "TUR", "Celtic": "SCO",
    "Liverpool": "ENG", "Union Berlin": "GER", "Antwerp": "BEL", "Lens": "FRA"
}

# Ergänze die Herkunftsländer der Teams im DataFrame (für spätere UEFA-Regelprüfung bei Auslosung)
df_pots["Country"] = df_pots["Club"].map(club_to_country)

In [57]:
# Versuche bis zu max_attempts-mal eine gültige Gruppenauslosung zu erstellen
max_attempts = 1000  # Sicherheitsschleife, falls keine gültige Kombination gefunden wird

for attempt in range(max_attempts):
    try:
        # Leere Gruppen initialisieren (A–H)
        groups = {f"Group {chr(65+i)}": [] for i in range(8)}

        # Schleife über die vier Töpfe
        for pot_num in range(1, 5):
            pot_teams = df_pots[df_pots["Pot"] == pot_num].copy()
            pot_teams = pot_teams.sample(frac=1).to_dict("records")  # Teams im Topf zufällig mischen

            for team in pot_teams:
                assigned = False
                group_names = list(groups.keys())
                random.shuffle(group_names)  # Gruppen zufällig durchgehen

                for group_name in group_names:
                    existing_countries = [t["Country"] for t in groups[group_name]]
                    
                    # Bedingung: Max. 1 Team pro Land und Topfgröße nicht überschreiten
                    if len(groups[group_name]) < pot_num and team["Country"] not in existing_countries:
                        groups[group_name].append(team)
                        assigned = True
                        break  # Team erfolgreich zugewiesen

                if not assigned:
                    # Kein passender Platz → Auslosung ungültig → Neustart
                    raise ValueError(f"Kein gültiger Platz für {team['Club']}")

        # Wenn alle Zuweisungen gültig waren → Abbruch der Schleife
        break

    except ValueError:
        continue  # Wiederhole mit neuer zufälliger Reihenfolge

else:
    # Nach max_attempts keine gültige Auslosung → Fehler
    raise RuntimeError("Es konnte keine gültige Gruppenauslosung erstellt werden.")

# Umwandlung in DataFrame zur Weiterverarbeitung
group_assignments = []
for group, teams in groups.items():
    for team in teams:
        group_assignments.append({
            "Group": group,
            "Club": team["Club"],
            "Country": team["Country"],
            "Pot": team["Pot"],
            "Elo": team["Elo"]
        })

df_group_draw = pd.DataFrame(group_assignments).sort_values(by=["Group", "Pot"]).reset_index(drop=True)

# Ausgabe zur Kontrolle
print("UEFA-konforme Gruppenauslosung erfolgreich abgeschlossen:")
display(df_group_draw)

UEFA-konforme Gruppenauslosung erfolgreich abgeschlossen:


,Group,Club,Country,Pot,Elo
0,Group A,Napoli,ITA,1,1892
1,Group A,RB Leipzig,GER,2,1841
2,Group A,Shakhtar,UKR,3,1591
3,Group A,Antwerp,BEL,4,1656
4,Group B,Benfica,POR,1,1806
5,Group B,Dortmund,GER,2,1841
6,Group B,Milan,ITA,3,1841
7,Group B,Lens,FRA,4,1709
8,Group C,Sevilla,ESP,1,1733
9,Group C,Porto,POR,2,1825


In [58]:
random.seed(42)
np.random.seed(42)
# Geschätzte Koeffizienten aus Poisson-Modell (basierend auf vorherigem Training)
BETA_0 = 0.2289
BETA_ELO = 0.0020
BETA_HOME = 0.2019

# Erwartete Tore auf Basis von Elo-Differenz und Heimvorteil
def expected_goals(elo_diff, home):
    return math.exp(BETA_0 + BETA_ELO * elo_diff + BETA_HOME * home)

# Simuliere Gruppenphase (Hin- und Rückspiel zwischen allen Teams)
def simulate_group_stage(df_group_draw):
    results = []
    for group in df_group_draw["Group"].unique():
        teams = df_group_draw[df_group_draw["Group"] == group].reset_index(drop=True)
        for i in range(4):
            for j in range(i+1, 4):
                team1, team2 = teams.iloc[i], teams.iloc[j]
                
                # Hinspiel: team1 ist Heimteam
                lam1 = expected_goals(team1["Elo"] - team2["Elo"], 1)
                lam2 = expected_goals(team2["Elo"] - team1["Elo"], 0)
                g1 = np.random.poisson(lam1)
                g2 = np.random.poisson(lam2)
                results.append((group, team1["Club"], team2["Club"], g1, g2))
                
                # Rückspiel: team2 ist Heimteam
                lam1 = expected_goals(team2["Elo"] - team1["Elo"], 1)
                lam2 = expected_goals(team1["Elo"] - team2["Elo"], 0)
                g1 = np.random.poisson(lam1)
                g2 = np.random.poisson(lam2)
                results.append((group, team2["Club"], team1["Club"], g1, g2))
    
    return pd.DataFrame(results, columns=["Group", "Home", "Away", "Goals Home", "Goals Away"])

# Punkte & Tordifferenz in der Gruppenphase berechnen
def compute_group_points(matches):
    points = {}
    goal_diff = {}
    for _, row in matches.iterrows():
        h, a = row["Home"], row["Away"]
        gh, ga = row["Goals Home"], row["Goals Away"]

        # Initialisiere falls Team noch nicht in Dict ist
        points[h] = points.get(h, 0)
        points[a] = points.get(a, 0)
        goal_diff[h] = goal_diff.get(h, 0)
        goal_diff[a] = goal_diff.get(a, 0)

        # Punktevergabe
        if gh > ga:
            points[h] += 3
        elif gh < ga:
            points[a] += 3
        else:
            points[h] += 1
            points[a] += 1

        # Tordifferenz aktualisieren
        goal_diff[h] += gh - ga
        goal_diff[a] += ga - gh

    return points, goal_diff

# K.O.-Runde (z. B. Achtel-, Viertel-, Halbfinale): Heim- & Rückspiel
def simulate_knockout_round(teams, df_group_draw):
    random.shuffle(teams)
    next_round = []
    for i in range(0, len(teams), 2):
        t1, t2 = teams[i], teams[i+1]
        elo1 = df_group_draw[df_group_draw["Club"] == t1]["Elo"].values[0]
        elo2 = df_group_draw[df_group_draw["Club"] == t2]["Elo"].values[0]

        # Hinspiel: t1 Heim
        g1 = np.random.poisson(expected_goals(elo1 - elo2, 1))
        g2 = np.random.poisson(expected_goals(elo2 - elo1, 0))

        # Rückspiel: t2 Heim
        g3 = np.random.poisson(expected_goals(elo2 - elo1, 1))
        g4 = np.random.poisson(expected_goals(elo1 - elo2, 0))

        total1 = g1 + g4
        total2 = g2 + g3

        # Sieger basierend auf Gesamttoren
        if total1 > total2:
            next_round.append(t1)
        elif total2 > total1:
            next_round.append(t2)
        else:
            next_round.append(random.choice([t1, t2]))  # Zufall bei Gleichstand

    return next_round

# Finale simulieren – 1 Spiel auf neutralem Boden (kein Heimvorteil)
def simulate_final(team1, team2, df_group_draw):
    elo1 = df_group_draw[df_group_draw["Club"] == team1]["Elo"].values[0]
    elo2 = df_group_draw[df_group_draw["Club"] == team2]["Elo"].values[0]
    g1 = np.random.poisson(expected_goals(elo1 - elo2, 0))
    g2 = np.random.poisson(expected_goals(elo2 - elo1, 0))

    if g1 > g2:
        return team1
    elif g2 > g1:
        return team2
    return random.choice([team1, team2])  # Zufall bei Gleichstand

# ================================ Haupt-Simulation ================================

winner_counts = Counter()

# Simulation des gesamten Turniers 10.000 Mal
for sim in range(10000):
    # Gruppenphase simulieren
    group_matches = simulate_group_stage(df_group_draw)
    pts, gd = compute_group_points(group_matches)

    # Punktestand und Tordifferenz in Tabelle übernehmen
    teams_in_groups = df_group_draw.copy()
    teams_in_groups["Points"] = teams_in_groups["Club"].map(pts).fillna(0)
    teams_in_groups["GD"] = teams_in_groups["Club"].map(gd).fillna(0)

    # Top 2 jeder Gruppe kommen weiter
    qualified = []
    for group in teams_in_groups["Group"].unique():
        top2 = (
            teams_in_groups[teams_in_groups["Group"] == group]
            .sort_values(by=["Points", "GD"], ascending=False)
            .head(2)["Club"]
            .tolist()
        )
        qualified.extend(top2)

    # K.O.-Phase simulieren
    qf = simulate_knockout_round(qualified, df_group_draw)
    sf = simulate_knockout_round(qf, df_group_draw)
    finalists = simulate_knockout_round(sf, df_group_draw)
    winner = simulate_final(finalists[0], finalists[1], df_group_draw)

    # Gewinner speichern
    winner_counts[winner] += 1

# Turniergewinner zusammenfassen & anzeigen
results_df = pd.DataFrame.from_dict(winner_counts, orient="index", columns=["Wins"]).sort_values(by="Wins", ascending=False)
print("🏆 Final Winners after 10,000 Simulations:")
display(results_df)

KeyboardInterrupt: 

### Python Version

In [1]:
import sys
import platform

print("Python-Version:", sys.version)
print("System:", platform.platform())


Python-Version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
System: Windows-10-10.0.26100-SP0


In [2]:
!pip show notebook


Name: notebook
Version: 6.5.4
Summary: A web-based notebook environment for interactive computing
Home-page: http://jupyter.org
Author: Jupyter Development Team
Author-email: jupyter@googlegroups.com
License: BSD
Location: C:\Users\charb\anaconda3\Lib\site-packages
Requires: argon2-cffi, ipykernel, ipython-genutils, jinja2, jupyter-client, jupyter-core, nbclassic, nbconvert, nbformat, nest-asyncio, prometheus-client, pyzmq, Send2Trash, terminado, tornado, traitlets
Required-by: jupyter, jupyterlab
